In [ ]:
# GDP Forecasting with ARIMA

This notebook forecasts GDP for selected countries using ARIMA models. Functions for data loading, plotting, and Excel output are included in a single cell.


import warnings
# Suppress FutureWarnings globally
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=r".*force_all_finite.*")
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import os
sys.path.append(str(Path(__file__).resolve().parent.parent / 'EIAfunctions'))
from func_clean_world_bank_data import clean_world_bank_data
from sklearn.metrics import mean_squared_error
import pmdarima as pm
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import PatternFill, Alignment, Font, Border, Side
import openpyxl

# --- All function definitions go here ---
def get_worldbank_gdp_data(plot_flag):
    # Full function code as in your script
    ...

def plot_gdp_forecast(worldbank_gdp_data, forecast_gdp, countries, title):
    ...

def create_excel_file_with_title(year: str, filename: str = "output.xlsx") -> int:
    ...

def append_styled_matrix_to_excel(df, matrix_name, year: str, start_col: int, filename: str = "output.xlsx", title_size=3) -> int:
    ...

def append_styled_series_to_excel(series: pd.Series, series_name, year: str, start_col: int, filename: str = "output.xlsx") -> int:
    ...


## Load GDP Data

We first load the World Bank GDP data for the selected countries, rename the year column, and set it as the index.


In [ ]:
# 01 upload gdp data
worldbank_gdp_data = get_worldbank_gdp_data(False)
worldbank_gdp_data.rename(columns={'Time': 'year'}, inplace=True)
worldbank_gdp_data['year'] = worldbank_gdp_data['year'].astype(int)
worldbank_gdp_data.set_index('year', inplace=True)


## Train-Test Split

We split the data into training and test sets, using 80% for training and 20% for testing.


In [ ]:
train_test_split = 0.8
train_size = int(len(worldbank_gdp_data) * train_test_split)
train, test = worldbank_gdp_data[:train_size], worldbank_gdp_data[train_size:]


In [ ]:
forecast_horizon = 16  # years ahead to predict
forecast_years = list(range(worldbank_gdp_data.index.max() + 1,
                            worldbank_gdp_data.index.max() + forecast_horizon + 1))
countries = ['ITA','JPN','CAN','FRA','DEU','GBR','USA']
forecast_gdp = pd.DataFrame(index=pd.Index(forecast_years, dtype=int), columns=countries)
    

## Fit ARIMA and Forecast

For each country, we select the best ARIMA(p,d,q) model using `auto_arima`, forecast the test set to evaluate accuracy, refit on the full series, and forecast 16 years ahead. The selected ARIMA orders are stored for reference.


In [ ]:
arima_orders = {}
for country in countries:
    print(f"\nProcessing {country}...")
    
    # Extract series
    train_series = worldbank_gdp_data[:train_size][country]
    test_series = worldbank_gdp_data[train_size:][country]
    
    # Fit ARIMA model (fixed d)
    d_value = 2  # or set to None for automatic selection
    model = pm.auto_arima(
        train_series,
        start_p=0, max_p=5,
        start_q=0, max_q=5,
        d=d_value,
        seasonal=False,
        stepwise=False,
        suppress_warnings=True,
        error_action='ignore',
        information_criterion='aic',
        n_jobs=-1
    )
    
    print(f"Selected ARIMA order for {country}: {model.order}")
    
    # Forecast on test
    prediction_on_test = model.predict(n_periods=len(test_series))
    mse = mean_squared_error(test_series, prediction_on_test)
    print(f"Test MSE for {country}: {mse:.2e}")
    
    # Refit on full series
    model.fit(worldbank_gdp_data[country])
    arima_orders[country] = model.order
    
    # Forecast 16 years ahead
    forecast_values = model.predict(n_periods=forecast_horizon)
    forecast_values.index = forecast_years
    forecast_gdp[country] = forecast_values


## Plot and Output

Plot the forecast and optionally save the results to Excel. This cell does not require explanation.


In [ ]:
plot_gdp_forecast(worldbank_gdp_data, forecast_gdp, countries, title=f"ARIMA GDP d={d_value}")

print("\nSelected ARIMA orders for all countries:")
for country, order in arima_orders.items():
    print(f"{country}: ARIMA{order}")

# Optional: save to Excel
if 0:
    gdp = pd.concat([worldbank_gdp_data, forecast_gdp])
    gdp.to_csv("Bench_predictions/gdp_ARIMAgdp_currentUSD04.csv", index=True)
